# VEYA v2 · Kaggle — обучение ConvNeXt и ViT одновременно (2×T4)
Тот же код, что и в Colab. Данные готовятся заново из тех же Kaggle-датасетов (тот же детерминированный сплит
по пациентам), затем две модели учатся параллельно — каждая на своей видеокарте.
Запускать через **Save Version → Save & Run All** (идёт в фоне до 12 ч, ноутбук можно закрыть).
Результат — в Output версии: `veya_out/models/*/best.pt`, `veya_out/preds/*.npz`, логи.

In [ ]:
import os, sys
CODE = '/kaggle/working/code'
os.makedirs(f'{CODE}/veya_v2', exist_ok=True)
open(f'{CODE}/veya_v2/__init__.py', 'w').close()
sys.path.insert(0, CODE)
DATA = '/tmp/veya_data'          # обработанные снимки (быстрый локальный диск)
PARTS = '/tmp/veya_parts'        # промежуточные файлы подготовки
OUT = '/kaggle/working/veya_out' # модели и предсказания — сохраняются в Output версии
os.makedirs(OUT, exist_ok=True)

In [ ]:
!pip -q install -U timm kagglehub
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
%%writefile /kaggle/working/code/veya_v2/data_prep.py
"""VEYA v2 — сборка единого датасета глазного дна.

Что делает:
  1. Скачивает публичные датасеты (Kaggle через kagglehub, без токена).
  2. Читает метки и приводит их к одной схеме на КАЖДЫЙ ГЛАЗ:
        dr_grade  0..4  (стадия ДР, -1 = неизвестно)
        dr        0/1   (есть ли ДР, -1 = неизвестно)
        glaucoma  0/1   (-1 = неизвестно / подозрение)
        cataract  0/1   (-1 = неизвестно)
  3. Обрезает чёрные поля, приводит к 512x512, считает перцептивный хэш.
  4. Находит дубликаты между датасетами и сливает их метки.
  5. Делит по ПАЦИЕНТАМ (а не по снимкам): train / val / test,
     плюс test_ext — целые датасеты, которые модель вообще не видит (IDRiD, PAPILA, RIM-ONE).
  6. Сохраняет на Google Drive по частям (parts/*.tar) + labels.csv.
"""
from __future__ import annotations

import hashlib
import os
import re
import shutil
import tarfile
import time
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:  # pragma: no cover
    def tqdm(x, **k):
        return x

IMG_SIZE = 512
LABEL_COLS = ["dr_grade", "dr", "glaucoma", "cataract"]
IMG_EXT = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

# Kaggle-источники (все публичные, скачиваются без токена)
SOURCES = {
    "eyepacs_aptos": "benjaminwarner/resized-2015-2019-blindness-detection-images",  # EyePACS 2015 (train+test с метками) + APTOS 2019
    "ddr": "samriddhibagchi/ddr-dataset-credits-to-authors",
    "idrid": "aaryapatel98/indian-diabetic-retinopathy-image-dataset",
    "odir": "andrewmvd/ocular-disease-recognition-odir5k",
    "smdg": "deathtrooper/multichannel-glaucoma-benchmark-dataset",  # REFUGE, RIM-ONE, PAPILA, ORIGA, G1020, ACRIMA ...
    "airogs": "deathtrooper/glaucoma-dataset-eyepacs-airogs-light-v2",
}

# Датасеты, которые целиком уходят во внешний тест (модель их не видит при обучении)
EXTERNAL_DATASETS = {"idrid"}
EXTERNAL_SMDG_PREFIXES = ("PAPILA", "RIM-ONE", "RIMONE")

# при слиянии дубликатов: чьи метки главнее
DATASET_PRIORITY = ["idrid", "odir", "ddr", "aptos", "eyepacs", "smdg", "airogs"]


# ----------------------------------------------------------------------------- utils
def _rec(path, dataset, image_id, patient, eye="", dr_grade=-1, dr=-1, glaucoma=-1, cataract=-1, subset=""):
    dr_grade = int(dr_grade)
    dr = int(dr)
    if dr_grade >= 0 and dr < 0:
        dr = int(dr_grade >= 1)
    return dict(
        src=str(path), dataset=dataset, subset=subset or dataset,
        image_id=f"{dataset}/{image_id}", patient=f"{dataset}/{patient}", eye=str(eye),
        dr_grade=dr_grade, dr=dr, glaucoma=int(glaucoma), cataract=int(cataract),
    )


def list_images(root) -> list[Path]:
    out = []
    for dp, _, fns in os.walk(root):
        for fn in fns:
            if Path(fn).suffix.lower() in IMG_EXT:
                out.append(Path(dp) / fn)
    return out


def find_files(root, name_regex) -> list[Path]:
    rx = re.compile(name_regex, re.I)
    out = []
    for dp, _, fns in os.walk(root):
        for fn in fns:
            if rx.search(fn):
                out.append(Path(dp) / fn)
    return sorted(out)


def show_tree(root, max_lines=40):
    """Печатает структуру папок (для отладки, если датасет поменял формат)."""
    root = Path(root)
    lines = 0
    for dp, dns, fns in os.walk(root):
        depth = len(Path(dp).relative_to(root).parts)
        if depth > 3:
            continue
        print("  " * depth + Path(dp).name + f"/  ({len(fns)} files)")
        lines += 1
        if lines >= max_lines:
            print("  ...")
            return


# ----------------------------------------------------------------------------- loaders
def load_eyepacs_aptos(root) -> list[dict]:
    imgs = list_images(root)
    by_stem = {}
    for p in imgs:
        by_stem.setdefault(p.stem.lower(), p)
    recs, miss = [], 0
    for csv in find_files(root, r"^(train|test)Labels15\.csv$"):
        df = pd.read_csv(csv)
        for im, lv in zip(df["image"].astype(str), df["level"]):
            p = by_stem.get(im.lower())
            if p is None:
                miss += 1
                continue
            pid, eye = (im.split("_") + [""])[:2]
            recs.append(_rec(p, "eyepacs", im, pid, eye, dr_grade=lv))
    for csv in find_files(root, r"^trainLabels19\.csv$"):
        df = pd.read_csv(csv)
        for im, lv in zip(df["id_code"].astype(str), df["diagnosis"]):
            p = by_stem.get(im.lower())
            if p is None:
                miss += 1
                continue
            recs.append(_rec(p, "aptos", im, im, "", dr_grade=lv))
    print(f"  eyepacs+aptos: {len(recs)} records, {miss} missing images")
    return recs


def load_ddr(root) -> list[dict]:
    recs, miss, ungradable = [], 0, 0
    txts = [t for t in find_files(root, r"^(train|valid|test)\.txt$") if "grading" in str(t).lower()]
    for txt in txts:
        img_dir = txt.parent / txt.stem
        by_name = {p.name.lower(): p for p in list_images(img_dir)} if img_dir.exists() else {}
        for line in Path(txt).read_text().splitlines():
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            name, grade = parts[0], int(parts[1])
            if grade > 4:  # 5 = ungradable
                ungradable += 1
                continue
            p = by_name.get(name.lower())
            if p is None:
                miss += 1
                continue
            stem = Path(name).stem
            recs.append(_rec(p, "ddr", stem, stem, "", dr_grade=grade))
    print(f"  ddr: {len(recs)} records, {miss} missing, {ungradable} ungradable skipped")
    return recs


def load_idrid(root) -> list[dict]:
    recs, miss = [], 0
    imgs = [p for p in list_images(root) if "grading" in str(p).lower()]
    for csv in find_files(root, r"\.csv$"):
        if "grading" not in str(csv).lower():
            continue
        df = pd.read_csv(csv)
        df.columns = [c.strip() for c in df.columns]
        if "Image name" not in df.columns or "Retinopathy grade" not in df.columns:
            continue
        df = df.dropna(subset=["Image name"])
        is_train = "train" in csv.name.lower()
        pool = [p for p in imgs if ("train" in str(p.parent).lower()) == is_train]
        by_stem = {p.stem.lower(): p for p in pool}
        tag = "tr" if is_train else "te"
        for im, g in zip(df["Image name"].astype(str), df["Retinopathy grade"]):
            p = by_stem.get(im.strip().lower())
            if p is None:
                miss += 1
                continue
            recs.append(_rec(p, "idrid", f"{tag}_{im}", f"{tag}_{im}", "", dr_grade=int(g)))
    print(f"  idrid: {len(recs)} records, {miss} missing")
    return recs


def _odir_eye_labels(kw: str):
    """Метки одного глаза из ODIR 'Diagnostic Keywords' (а не общие метки пациента)."""
    k = str(kw).lower()
    parts = [x.strip() for x in re.split(r"[，,;]", k) if x.strip()]
    if any("no fundus image" in x for x in parts):
        return None
    grade, dr = 0, 0
    for x in parts:
        if "proliferative" in x or "diabetic retinopathy" in x:
            dr = 1
            if "mild" in x:
                g = 1
            elif "moderate" in x:
                g = 2
            elif "severe" in x and "non" in x:
                g = 3
            elif "proliferative" in x and "non" not in x:
                g = 4
            else:
                g = -1
            grade = g if (grade == 0 or g > grade) else grade
            if g == -1 and grade == 0:
                grade = -1
    glau = 0
    if any("glaucoma" in x and "suspect" not in x for x in parts):
        glau = 1
    elif any("glaucoma" in x for x in parts):
        glau = -1
    cat = int(any("cataract" in x for x in parts))
    return grade, dr, glau, cat


def load_odir(root) -> list[dict]:
    imgs = list_images(root)
    train_imgs = [p for p in imgs if "training" in str(p).lower()] or imgs
    by_name = {}
    for p in train_imgs:
        by_name.setdefault(p.name.lower(), p)
    # data.xlsx (оригинал) или full_df.csv (копия Kaggle)
    df = None
    for f in find_files(root, r"^data\.xlsx$"):
        df = pd.read_excel(f)
        break
    if df is None:
        f = find_files(root, r"^full_df\.csv$")[0]
        df = pd.read_csv(f)
    df = df.drop_duplicates(subset=["ID"])
    recs, miss, skipped = [], 0, 0
    for _, r in df.iterrows():
        for side, eye in (("Left", "left"), ("Right", "right")):
            fn = str(r[f"{side}-Fundus"])
            lab = _odir_eye_labels(r[f"{side}-Diagnostic Keywords"])
            if lab is None:
                skipped += 1
                continue
            p = by_name.get(fn.lower())
            if p is None:
                miss += 1
                continue
            g, d, gl, c = lab
            recs.append(_rec(p, "odir", Path(fn).stem, r["ID"], eye, dr_grade=g, dr=d, glaucoma=gl, cataract=c))
    print(f"  odir: {len(recs)} records (per-eye labels), {miss} missing, {skipped} no-fundus skipped")
    return recs


def load_smdg(root) -> list[dict]:
    meta = None
    for f in find_files(root, r"\.csv$"):
        try:
            d = pd.read_csv(f)
        except Exception:
            continue
        if {"names", "types"} <= set(d.columns):
            meta = d
            break
    assert meta is not None, "SMDG: не найден metadata csv с колонками names/types"
    imgs = [p for p in list_images(root) if "full-fundus" in str(p).lower()]
    by_stem = {p.stem.lower(): p for p in imgs}
    recs, miss = [], 0
    for name, t in zip(meta["names"].astype(str), meta["types"]):
        if Path(name).suffix.lower() in IMG_EXT:
            name = Path(name).stem
        p = by_stem.get(name.lower())
        if p is None:
            miss += 1
            continue
        prefix = re.split(r"-\d|_\d", name)[0]
        g = int(t) if int(t) in (0, 1) else -1  # -1 = подозрение
        recs.append(_rec(p, "smdg", name, name, "", glaucoma=g, subset=f"smdg:{prefix}"))
    c = Counter(r["subset"] for r in recs)
    print(f"  smdg: {len(recs)} records, {miss} missing; sources: {dict(c.most_common())}")
    return recs


def load_airogs(root) -> list[dict]:
    recs = []
    for p in list_images(root):
        lab = p.parent.name.upper()
        if lab not in ("RG", "NRG"):
            continue
        recs.append(_rec(p, "airogs", p.stem, p.stem, "", glaucoma=int(lab == "RG")))
    print(f"  airogs: {len(recs)} records")
    return recs


LOADERS = {
    "eyepacs_aptos": load_eyepacs_aptos,
    "ddr": load_ddr,
    "idrid": load_idrid,
    "odir": load_odir,
    "smdg": load_smdg,
    "airogs": load_airogs,
}


# ----------------------------------------------------------------------------- image processing
def crop_fundus(img: np.ndarray, size: int = IMG_SIZE):
    """Обрезка чёрных полей + центрирование круга сетчатки в квадрат size x size."""
    if img is None or img.size == 0:
        return None
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    thr = max(10, float(np.percentile(gray, 5)) + 8)
    mask = gray > thr
    rows = np.where(mask.mean(1) > 0.02)[0]
    cols = np.where(mask.mean(0) > 0.02)[0]
    if len(rows) < h * 0.2 or len(cols) < w * 0.2:
        y0, y1, x0, x1 = 0, h, 0, w  # маска не нашлась — берём целиком
    else:
        y0, y1, x0, x1 = rows[0], rows[-1] + 1, cols[0], cols[-1] + 1
    img = img[y0:y1, x0:x1]
    h, w = img.shape[:2]
    s = max(h, w)
    canvas = np.zeros((s, s, 3), dtype=img.dtype)
    canvas[(s - h) // 2:(s - h) // 2 + h, (s - w) // 2:(s - w) // 2 + w] = img
    return cv2.resize(canvas, (size, size), interpolation=cv2.INTER_AREA)


def fingerprint(img512: np.ndarray):
    """Перцептивный хэш центральной части (без чёрного круга) + маленькая high-pass миниатюра."""
    g = cv2.cvtColor(img512, cv2.COLOR_BGR2GRAY).astype(np.float32)
    s = g.shape[0]
    c = g[int(s * 0.2):int(s * 0.8), int(s * 0.2):int(s * 0.8)]
    small = cv2.resize(c, (64, 64), interpolation=cv2.INTER_AREA)
    d = cv2.dct(small)[:8, :8].flatten()
    med = np.median(d[1:])
    d[0] = med  # DC-компоненту (общая яркость) не учитываем
    h = 0
    for b in (d > med):
        h = (h << 1) | int(b)
    # миниатюра сосудистого рисунка: high-pass + обрезка выбросов (чтобы яркий диск не доминировал)
    c2 = g[int(s * 0.15):int(s * 0.85), int(s * 0.15):int(s * 0.85)]
    t = cv2.resize(c2, (48, 48), interpolation=cv2.INTER_AREA)
    t = t - cv2.GaussianBlur(t, (0, 0), 1.5)
    t = np.clip((t - t.mean()) / (t.std() + 1e-6), -1.5, 1.5)
    return np.uint64(h), t.astype(np.float16)


def _process_one(args):
    src, dst = args
    try:
        img = cv2.imread(src, cv2.IMREAD_COLOR)
        if img is None:
            return None
        h, w = img.shape[:2]
        if max(h, w) > 2 * IMG_SIZE:  # ускоряем: сначала грубо уменьшаем
            f = 2 * IMG_SIZE / max(h, w)
            img = cv2.resize(img, (int(w * f), int(h * f)), interpolation=cv2.INTER_AREA)
        out = crop_fundus(img)
        if out is None:
            return None
        if out.mean() < 8:  # пустой/чёрный кадр
            return None
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        cv2.imwrite(dst, out, [cv2.IMWRITE_JPEG_QUALITY, 92])
        ph, th = fingerprint(out)
        return int(ph), th
    except Exception:
        return None


def process_images(recs: list[dict], out_dir: str, workers: int | None = None) -> pd.DataFrame:
    workers = workers or os.cpu_count() or 2
    jobs = []
    for r in recs:
        rel = f"img/{r['image_id']}.jpg".replace(":", "_")
        r["path"] = rel
        jobs.append((r["src"], os.path.join(out_dir, rel)))
    keep, hashes, thumbs = [], [], []
    with ProcessPoolExecutor(workers) as ex:
        for r, res in tqdm(zip(recs, ex.map(_process_one, jobs, chunksize=32)), total=len(jobs), desc="images"):
            if res is None:
                continue
            keep.append(r)
            hashes.append(res[0])
            thumbs.append(res[1])
    df = pd.DataFrame(keep).drop(columns=["src"])
    df["phash"] = np.array(hashes, dtype=np.uint64).astype(np.int64)  # int64 для csv
    print(f"  processed {len(df)}/{len(recs)} images")
    return df, (np.stack(thumbs) if thumbs else np.zeros((0, 48, 48), np.float16))


# ----------------------------------------------------------------------------- dedup + split
class DSU:
    def __init__(self, n):
        self.p = np.arange(n)

    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]
            a = self.p[a]
        return a

    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[max(a, b)] = min(a, b)


def popcount64(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.uint64)
    if hasattr(np, "bitwise_count"):  # numpy >= 2.0
        return np.bitwise_count(x).astype(np.int64)
    c = np.zeros(x.shape, dtype=np.int64)
    for i in range(8):
        c += np.unpackbits(((x >> np.uint64(8 * i)) & np.uint64(255)).astype(np.uint8)[..., None], axis=-1).sum(-1)
    return c


def find_duplicates(phash: np.ndarray, thumbs: np.ndarray, max_ham=10, min_corr=0.85, max_bucket=800,
                    chunk=20000):
    """Кандидаты — совпадение одного из 4 кусков по 16 бит (гарантирует нахождение пар с hamming<=3,
    большинство пар до ~8), подтверждение — корреляция high-pass миниатюр.
    Пары проверяются кусками, чтобы не съесть всю RAM Colab."""
    ph = phash.astype(np.uint64)
    n = len(ph)
    flat = thumbs.reshape(n, -1)  # float16, ~0.6 GB на 130k снимков
    norm = np.sqrt(np.einsum("ij,ij->i", flat, flat, dtype=np.float32)) + 1e-6
    pairs = set()
    big, cand = 0, 0
    for k in range(4):
        key = ((ph >> np.uint64(16 * k)) & np.uint64(0xFFFF)).astype(np.int64)
        order = np.argsort(key, kind="stable")
        ks = key[order]
        starts = np.flatnonzero(np.r_[True, ks[1:] != ks[:-1]])
        ends = np.r_[starts[1:], len(ks)]
        for s, e in zip(starts, ends):
            m = e - s
            if m < 2:
                continue
            if m > max_bucket:
                big += 1
                continue
            idx = order[s:e]
            x = ph[idx]
            ham = popcount64(x[:, None] ^ x[None, :])
            ii, jj = np.where(np.triu(ham <= max_ham, 1))
            cand += len(ii)
            for c in range(0, len(ii), chunk):
                a, b = idx[ii[c:c + chunk]], idx[jj[c:c + chunk]]
                corr = np.einsum("ij,ij->i", flat[a], flat[b], dtype=np.float32) / (norm[a] * norm[b])
                ok = corr >= min_corr
                for u, v in zip(a[ok], b[ok]):
                    pairs.add((int(min(u, v)), int(max(u, v))))
    print(f"  кандидатов проверено: {cand}, пропущено больших корзин: {big}")
    return sorted(pairs)


def merge_and_split(df: pd.DataFrame, thumbs: np.ndarray, seed=2026, val_frac=0.12, test_frac=0.10) -> pd.DataFrame:
    df = df.reset_index(drop=True)
    n = len(df)
    pairs = find_duplicates(df["phash"].values.astype(np.int64).view(np.uint64), thumbs)
    print(f"  near-duplicate pairs: {len(pairs)}")
    dsn = df["dataset"].values
    print("  пары дубликатов по датасетам:", dict(Counter(tuple(sorted((dsn[a], dsn[b]))) for a, b in pairs).most_common(12)))

    # 1) кластеры дубликатов → оставляем один снимок, сливаем метки
    dup = DSU(n)
    for a, b in pairs:
        dup.union(a, b)
    droot = np.array([dup.find(i) for i in range(n)])
    prio = {d: i for i, d in enumerate(DATASET_PRIORITY)}
    df["_prio"] = df["dataset"].map(lambda d: prio.get(d, 99))
    df["_ext"] = is_external(df)
    sizes = np.bincount(droot, minlength=n)
    keep_mask = sizes[droot] == 1
    removed = 0
    multi = defaultdict(list)
    for i in np.flatnonzero(sizes[droot] > 1):
        multi[droot[i]].append(i)
    for root, idxs in multi.items():
        sub = df.loc[idxs].sort_values(["_ext", "_prio"], ascending=[False, True])
        k = sub.index[0]
        for col in LABEL_COLS:
            if df.at[k, col] >= 0:
                continue
            vals = [v for v in sub[col].tolist() if v >= 0]
            if vals:
                df.at[k, col] = vals[0] if (len(set(vals)) == 1 or col == "dr_grade") else -1
        keep_mask[k] = True
        removed += len(idxs) - 1
    keep_idx = np.flatnonzero(keep_mask)
    fix = (df["dr"] < 0) & (df["dr_grade"] >= 0)
    df.loc[fix, "dr"] = (df.loc[fix, "dr_grade"] >= 1).astype(int)
    print(f"  removed {removed} duplicate images across/within datasets")

    # 2) группы для сплита = пациенты ∪ кластеры дубликатов
    g = DSU(n)
    for a, b in pairs:
        g.union(a, b)
    first_of_patient = {}
    for i, p in enumerate(df["patient"].values):
        if p in first_of_patient:
            g.union(i, first_of_patient[p])
        else:
            first_of_patient[p] = i
    groot = np.array([g.find(i) for i in range(n)])
    ext_groups = set(groot[df["_ext"].values])

    def bucket(root):
        h = int(hashlib.md5(f"{seed}-{root}".encode()).hexdigest()[:8], 16) / 0xFFFFFFFF
        return "test" if h < test_frac else ("val" if h < test_frac + val_frac else "train")

    df["split"] = [("test_ext" if r in ext_groups else bucket(r)) for r in groot]
    df["group"] = groot
    out = df.loc[sorted(keep_idx)].drop(columns=["_prio", "_ext"]).reset_index(drop=True)
    return out


def is_external(df: pd.DataFrame) -> np.ndarray:
    ext = df["dataset"].isin(EXTERNAL_DATASETS).values.copy()
    sub = df["subset"].astype(str).str.upper()
    for pfx in EXTERNAL_SMDG_PREFIXES:
        ext |= sub.str.startswith(f"SMDG:{pfx}").values
    return ext


def summarize(df: pd.DataFrame):
    print("\n=== Итоговый датасет ===")
    print(df.groupby(["split", "dataset"]).size().unstack(0).fillna(0).astype(int))
    for col in ["dr", "glaucoma", "cataract"]:
        t = df[df[col] >= 0].groupby("split")[col].agg(["count", "sum"])
        print(f"\n{col}: (count = размечено, sum = больных)\n{t}")
    t = df[df["dr_grade"] >= 0].groupby(["split", "dr_grade"]).size().unstack(1).fillna(0).astype(int)
    print(f"\ndr_grade:\n{t}")


# ----------------------------------------------------------------------------- pipeline
def run_all(work_dir="/content/veya_data", drive_dir="/content/drive/MyDrive/veya_v2/data",
            only: list[str] | None = None, delete_raw=True):
    """Каждый датасет после обработки сразу сохраняется на Drive (parts/<name>.tar + .csv + thumbs),
    поэтому если Colab оборвётся — при перезапуске готовые датасеты пропускаются."""
    import kagglehub

    work = Path(work_dir)
    work.mkdir(parents=True, exist_ok=True)
    parts = Path(drive_dir) / "parts"
    parts.mkdir(parents=True, exist_ok=True)
    failed = []

    for key, slug in SOURCES.items():
        if only and key not in only:
            continue
        if (parts / f"{key}.csv").exists():
            print(f"[{key}] уже на Drive, пропускаю")
            continue
        t0 = time.time()
        print(f"\n[{key}] скачиваю {slug} ...")
        try:
            root = kagglehub.dataset_download(slug)
            print(f"[{key}] скачано за {time.time()-t0:.0f}s")
            recs = LOADERS[key](root)
            if not recs:
                raise RuntimeError("0 записей")
        except Exception as e:
            print(f"[{key}] ОШИБКА: {e!r}\nСтруктура папки:")
            try:
                show_tree(root)
            except Exception:
                pass
            failed.append(key)
            continue
        df, thumbs = process_images(recs, str(work))
        # на Drive: tar картинок этого датасета + метки + отпечатки
        tmp = work / f"{key}.tar"
        with tarfile.open(tmp, "w") as tar:
            for p in df["path"]:
                tar.add(work / p, arcname=p)
        shutil.copy(tmp, parts / f"{key}.tar")
        os.remove(tmp)
        np.save(parts / f"{key}_thumbs.npy", thumbs)
        df.to_csv(parts / f"{key}.csv", index=False)  # csv последним = маркер «готово»
        if delete_raw:
            shutil.rmtree(root, ignore_errors=True)
            shutil.rmtree(Path.home() / ".cache" / "kagglehub", ignore_errors=True)
        print(f"[{key}] готово за {(time.time()-t0)/60:.1f} мин, {len(df)} снимков")

    if failed:
        print(f"\n⚠ Не получилось: {failed}. Остальное сохранено; можно повторить run_all(only={failed}).")
    return build_labels(drive_dir)


def build_labels(drive_dir="/content/drive/MyDrive/veya_v2/data"):
    """Склейка всех датасетов → дедупликация → сплит по пациентам → labels.csv на Drive."""
    parts = Path(drive_dir) / "parts"
    keys = [k for k in SOURCES if (parts / f"{k}.csv").exists()]
    df = pd.concat([pd.read_csv(parts / f"{k}.csv") for k in keys], ignore_index=True)
    thumbs = np.concatenate([np.load(parts / f"{k}_thumbs.npy") for k in keys])
    final = merge_and_split(df, thumbs)
    final.to_csv(Path(drive_dir) / "labels.csv", index=False)
    summarize(final)
    return final


def restore_from_drive(local_dir="/content/veya_data", drive_dir="/content/drive/MyDrive/veya_v2/data"):
    """Новая сессия Colab: распаковываем tar-ы с Drive на быстрый локальный диск."""
    local = Path(local_dir)
    local.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    for t in sorted((Path(drive_dir) / "parts").glob("*.tar")):
        flag = local / f".{t.stem}.done"
        if flag.exists():
            continue
        with tarfile.open(t) as tar:
            tar.extractall(local)
        flag.touch()
        print(f"  распакован {t.name}")
    shutil.copy(Path(drive_dir) / "labels.csv", local / "labels.csv")
    print(f"Данные готовы за {time.time()-t0:.0f}s")
    return local

In [ ]:
%%writefile /kaggle/working/code/veya_v2/train.py
"""VEYA v2 — обучение базовых моделей ансамбля (multi-task: стадия ДР, ДР, глаукома, катаракта).

Модели («глаза» ансамбля):
  effv2s   — EfficientNetV2-S, 512px  (CNN, ловит мелкие поражения: микроаневризмы, экссудаты)
  convnext — ConvNeXt-Small, 384px    (современная CNN, другие ошибки чем у EfficientNet)
  vit      — ViT-B/14 DINOv2, 224px   (трансформер, смотрит на снимок целиком)
  retfound — RETFound ViT-L, 224px    (foundation-модель, предобучена на 1.6 млн снимков сетчатки;
                                       нужен HF_TOKEN в секретах Colab, иначе используйте vit)

Всё устойчиво к обрывам Colab: чекпоинт на Google Drive каждые N минут, запуск заново — продолжает с места.
"""
from __future__ import annotations

import json
import math
import os
import random
import shutil
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, cohen_kappa_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset, Sampler

try:
    from tqdm.auto import tqdm
except ImportError:  # pragma: no cover
    def tqdm(x, **k):
        return x

BINARY = ["dr", "glaucoma", "cataract"]
HEADS = ["dr_grade"] + BINARY

MODELS = {
    "effv2s": dict(timm="tf_efficientnetv2_s.in21k_ft_in1k", size=512, bs=16, accum=2, lr=3e-4, epochs=14,
                   samples_per_epoch=36000, channels_last=True),
    "convnext": dict(timm="convnext_small.fb_in22k_ft_in1k_384", size=384, bs=16, accum=2, lr=1.5e-4, epochs=12,
                     samples_per_epoch=36000, channels_last=True, drop_path=0.2),
    "vit": dict(timm="vit_base_patch14_dinov2.lvd142m", size=224, bs=32, accum=1, lr=4e-5, epochs=12,
                samples_per_epoch=36000, kwargs=dict(img_size=224), drop_path=0.1, layer_decay=0.75),
    "retfound": dict(timm="vit_large_patch16_224", size=224, bs=16, accum=2, lr=3e-5, epochs=10,
                     samples_per_epoch=36000, kwargs=dict(global_pool="avg"), drop_path=0.2, layer_decay=0.65,
                     hf=("YukunZhou/RETFound_mae_natureCFP", "RETFound_mae_natureCFP.pth")),
}

MEAN = np.array([0.485, 0.456, 0.406], np.float32) * 255
STD = np.array([0.229, 0.224, 0.225], np.float32) * 255


# ============================================================================ data
def load_image(path: str, size: int) -> np.ndarray:
    flag = cv2.IMREAD_REDUCED_COLOR_2 if size <= 256 else cv2.IMREAD_COLOR
    img = cv2.imread(path, flag)
    if img is None:
        img = np.zeros((size, size, 3), np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if img.shape[0] != size:
        img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    return img


def augment(img: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Аугментации, безопасные для глазного дна (не ломают признаки болезней)."""
    s = img.shape[0]
    if rng.random() < 0.5:
        img = img[:, ::-1]
    if rng.random() < 0.3:
        img = img[::-1]
    ang = rng.uniform(-180, 180)
    scale = rng.uniform(0.92, 1.12)
    tx, ty = rng.uniform(-0.04, 0.04, 2) * s
    M = cv2.getRotationMatrix2D((s / 2, s / 2), ang, scale)
    M[:, 2] += (tx, ty)
    img = cv2.warpAffine(np.ascontiguousarray(img), M, (s, s), flags=cv2.INTER_LINEAR, borderValue=(0, 0, 0))
    img = img.astype(np.float32)
    # яркость / контраст / гамма / баланс каналов (разные камеры и смартфон-насадка)
    img = img * rng.uniform(0.75, 1.25) + rng.uniform(-20, 20)
    mean = img.mean()
    img = (img - mean) * rng.uniform(0.8, 1.25) + mean
    img *= rng.uniform(0.92, 1.08, size=3)[None, None, :]
    img = np.clip(img, 0, 255)
    if rng.random() < 0.3:
        g = rng.uniform(0.8, 1.25)
        img = 255.0 * (img / 255.0) ** g
    if rng.random() < 0.2:
        k = rng.choice([3, 5])
        img = cv2.GaussianBlur(img, (k, k), 0)
    if rng.random() < 0.15:
        img = img + rng.normal(0, rng.uniform(2, 8), img.shape)
    return np.clip(img, 0, 255)


class FundusDS(Dataset):
    def __init__(self, df: pd.DataFrame, root: str, size: int, train: bool):
        self.paths = [os.path.join(root, p) for p in df["path"].values]
        self.y = df[HEADS].values.astype(np.int64)
        self.size = size
        self.train = train

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = load_image(self.paths[i], self.size)
        if self.train:
            rng = np.random.default_rng((os.getpid() * 1_000_003 + i * 7919 + int(time.time() * 1e6)) % 2**32)
            img = augment(img, rng)
        else:
            img = img.astype(np.float32)
        img = ((img - MEAN) / STD).astype(np.float32)
        x = torch.from_numpy(np.ascontiguousarray(img.transpose(2, 0, 1)))
        return x, torch.from_numpy(self.y[i])


class EpochSampler(Sampler):
    """Взвешенная выборка фиксированного размера на эпоху (баланс датасетов и редких болезней).
    Детерминирована по номеру эпохи → после обрыва Colab продолжаем с того же шага."""

    def __init__(self, weights: np.ndarray, n: int, epoch: int, start: int = 0, seed: int = 2026):
        rng = np.random.default_rng(seed + epoch)
        p = weights / weights.sum()
        self.idx = rng.choice(len(weights), size=n, replace=True, p=p)[start:]

    def __iter__(self):
        return iter(self.idx.tolist())

    def __len__(self):
        return len(self.idx)


def sample_weights(df: pd.DataFrame) -> np.ndarray:
    """Вес снимка ~ 1/sqrt(размер его «страты»): страта = датасет + комбинация меток.
    Больные глаукомой/катарактой/тяжёлой ДР попадают в батчи заметно чаще."""
    key = (df["dataset"].astype(str) + "|" + df["dr_grade"].astype(str) + "|" +
           df["glaucoma"].astype(str) + "|" + df["cataract"].astype(str))
    cnt = key.map(key.value_counts())
    w = 1.0 / np.sqrt(cnt.values.astype(np.float64))
    # дополнительно поднимаем редкие позитивы
    w *= np.where(df["glaucoma"].values == 1, 1.5, 1.0)
    w *= np.where(df["cataract"].values == 1, 2.0, 1.0)
    w *= np.where(df["dr_grade"].values >= 3, 1.5, 1.0)
    return w


# ============================================================================ model
class VeyaNet(nn.Module):
    """Backbone + общий многослойный «нейрон-мозг» модели + 4 головы."""

    def __init__(self, name: str, pretrained=True, drop=0.3, drop_path=0.0, **kw):
        super().__init__()
        import timm
        extra = dict(kw)
        if drop_path:
            extra["drop_path_rate"] = drop_path
        self.backbone = timm.create_model(name, pretrained=pretrained, num_classes=0, **extra)
        d = self.backbone.num_features
        self.neck = nn.Sequential(
            nn.Dropout(drop), nn.Linear(d, 768), nn.GELU(),
            nn.Dropout(drop / 2), nn.Linear(768, 384), nn.GELU(),
        )
        self.heads = nn.ModuleDict({
            "dr_grade": nn.Linear(384, 5),
            **{k: nn.Linear(384, 1) for k in BINARY},
        })
        self.emb_dim = d

    def forward(self, x, return_emb=False):
        f = self.backbone(x)
        h = self.neck(f)
        out = {k: (v(h) if k == "dr_grade" else v(h).squeeze(1)) for k, v in self.heads.items()}
        return (out, f) if return_emb else out


def load_retfound(model: VeyaNet, repo: str, fname: str):
    from huggingface_hub import hf_hub_download
    token = os.environ.get("HF_TOKEN")
    path = hf_hub_download(repo, fname, token=token)
    ck = torch.load(path, map_location="cpu", weights_only=False)
    sd = ck.get("model", ck.get("teacher", ck))
    sd = {k.replace("module.", "").replace("backbone.", ""): v for k, v in sd.items()}
    sd = {k: v for k, v in sd.items() if not k.startswith(("decoder", "mask_token", "head"))}
    msg = model.backbone.load_state_dict(sd, strict=False)
    print(f"RETFound загружен. missing={len(msg.missing_keys)} {msg.missing_keys[:6]} "
          f"unexpected={len(msg.unexpected_keys)} {msg.unexpected_keys[:6]}")


def build_model(key: str, pretrained=True) -> VeyaNet:
    cfg = MODELS[key]
    m = VeyaNet(cfg["timm"], pretrained=pretrained and "hf" not in cfg,
                drop_path=cfg.get("drop_path", 0.0), **cfg.get("kwargs", {}))
    if pretrained and "hf" in cfg:
        load_retfound(m, *cfg["hf"])
    return m


def param_groups(model: VeyaNet, lr: float, wd=0.05, layer_decay: float | None = None):
    """Голова учится в 10 раз быстрее; для ViT — послойное затухание lr (нижние слои меняются меньше)."""
    head = list(model.neck.parameters()) + list(model.heads.parameters())
    groups = [dict(params=head, lr=lr * 10, weight_decay=wd)]
    bb = model.backbone
    if layer_decay and hasattr(bb, "blocks"):
        n = len(bb.blocks)
        layers = {}
        for name, p in bb.named_parameters():
            if name.startswith("blocks."):
                lid = int(name.split(".")[1]) + 1
            elif name.startswith(("cls_token", "pos_embed", "patch_embed", "reg_token")):
                lid = 0
            else:
                lid = n + 1
            no_wd = p.ndim == 1 or name.endswith(".bias") or "token" in name or "pos_embed" in name
            layers.setdefault((lid, no_wd), []).append(p)
        for (lid, no_wd), ps in layers.items():
            groups.append(dict(params=ps, lr=lr * layer_decay ** (n + 1 - lid), weight_decay=0.0 if no_wd else wd))
    else:
        dec, nodec = [], []
        for name, p in bb.named_parameters():
            (nodec if p.ndim == 1 or name.endswith(".bias") else dec).append(p)
        groups += [dict(params=dec, lr=lr, weight_decay=wd), dict(params=nodec, lr=lr, weight_decay=0.0)]
    for g in groups:
        g["base_lr"] = g["lr"]
    return groups


# ============================================================================ loss + metrics
def multitask_loss(out, y, pos_weight: dict, smoothing=0.05):
    losses = {}
    yg = y[:, 0]
    m = yg >= 0
    if m.any():
        losses["dr_grade"] = F.cross_entropy(out["dr_grade"][m].float(), yg[m], label_smoothing=smoothing)
    for j, k in enumerate(BINARY, start=1):
        yy = y[:, j]
        m = yy >= 0
        if m.any():
            t = yy[m].float() * (1 - smoothing) + smoothing / 2
            losses[k] = F.binary_cross_entropy_with_logits(
                out[k][m].float(), t, pos_weight=torch.tensor(pos_weight[k], device=t.device))
    total = sum(losses.values()) if losses else out["dr"].sum() * 0
    return total, {k: float(v.detach()) for k, v in losses.items()}


def compute_metrics(logits: dict, y: np.ndarray) -> dict:
    res = {}
    yg = y[:, 0]
    m = yg >= 0
    if m.sum() > 10:
        pred = logits["dr_grade"][m].argmax(1)
        res["dr_grade_acc"] = accuracy_score(yg[m], pred)
        res["dr_grade_qwk"] = cohen_kappa_score(yg[m], pred, weights="quadratic")
    for j, k in enumerate(BINARY, start=1):
        yy = y[:, j]
        m = yy >= 0
        if m.sum() > 10 and 0 < yy[m].sum() < m.sum():
            p = 1 / (1 + np.exp(-logits[k][m]))
            res[f"{k}_auc"] = roc_auc_score(yy[m], p)
            res[f"{k}_bal_acc"] = 0.5 * ((p[yy[m] == 1] >= 0.5).mean() + (p[yy[m] == 0] < 0.5).mean())
    keys = [res.get("dr_auc"), res.get("glaucoma_auc"), res.get("cataract_auc"), res.get("dr_grade_qwk")]
    keys = [k for k in keys if k is not None]
    res["score"] = float(np.mean(keys)) if keys else 0.0
    return res


# ============================================================================ checkpoint helpers
def atomic_save(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_local = Path("/tmp") / (path.name + ".tmp")
    torch.save(obj, tmp_local)
    tmp_remote = path.with_suffix(path.suffix + ".tmp")
    shutil.copy(tmp_local, tmp_remote)
    os.replace(tmp_remote, path)
    tmp_local.unlink(missing_ok=True)


class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)
            else:
                self.shadow[k].copy_(v)

    def copy_to(self, model):
        model.load_state_dict({k: v.to(dtype=model.state_dict()[k].dtype) for k, v in self.shadow.items()})


# ============================================================================ train / eval
@torch.no_grad()
def predict(model, df, root, size, bs=32, tta=True, device="cuda", return_emb=False, workers=2):
    model.eval()
    dl = DataLoader(FundusDS(df, root, size, train=False), batch_size=bs, shuffle=False,
                    num_workers=workers, pin_memory=True)
    outs = {k: [] for k in HEADS}
    embs = []
    for x, _ in tqdm(dl, desc="predict", leave=False):
        x = x.to(device, non_blocking=True)
        views = [x, torch.flip(x, dims=[3])] if tta else [x]
        acc, eacc = None, None
        for v in views:
            with torch.autocast(device_type="cuda" if device != "cpu" else "cpu", dtype=torch.float16,
                                enabled=device != "cpu"):
                o, e = model(v, return_emb=True)
            o = {k: t.float() for k, t in o.items()}
            acc = o if acc is None else {k: acc[k] + o[k] for k in o}
            eacc = e.float() if eacc is None else eacc + e.float()
        for k in HEADS:
            outs[k].append((acc[k] / len(views)).cpu().numpy())
        if return_emb:
            embs.append((eacc / len(views)).cpu().numpy().astype(np.float16))
    res = {k: np.concatenate(v) for k, v in outs.items()}
    if return_emb:
        res["emb"] = np.concatenate(embs)
    return res


def train_model(key: str, data_dir="/content/veya_data", drive_dir="/content/drive/MyDrive/veya_v2",
                epochs: int | None = None, ckpt_minutes=12, workers=2, max_hours: float | None = None,
                quick_test=False, pretrained=True):
    cfg = dict(MODELS[key])
    if epochs:
        cfg["epochs"] = epochs
    device = "cuda" if torch.cuda.is_available() else "cpu"
    out_dir = Path(drive_dir) / "models" / key
    out_dir.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(Path(data_dir) / "labels.csv")
    tr = df[df.split == "train"].reset_index(drop=True)
    va = df[df.split == "val"].reset_index(drop=True)
    if quick_test:
        tr, va = tr.head(64), va.head(32)
        cfg.update(samples_per_epoch=32, bs=8, accum=1)
    print(f"[{key}] train={len(tr)} val={len(va)} size={cfg['size']} device={device}")

    pos_weight = {}
    for k in BINARY:
        y = tr[k][tr[k] >= 0]
        pos = max(1, int(y.sum()))
        pos_weight[k] = float(min(5.0, math.sqrt((len(y) - pos) / pos)))
    print("pos_weight:", pos_weight)

    model = build_model(key, pretrained=pretrained and not quick_test).to(device)
    if cfg.get("channels_last") and device == "cuda":
        model = model.to(memory_format=torch.channels_last)
    opt = torch.optim.AdamW(param_groups(model, cfg["lr"], layer_decay=cfg.get("layer_decay")), betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")
    ema = EMA(model, decay=0.999 if not quick_test else 0.5)
    steps_per_epoch = cfg["samples_per_epoch"] // (cfg["bs"] * cfg["accum"])
    total_steps = steps_per_epoch * cfg["epochs"]
    warmup = max(1, int(0.03 * total_steps))

    def lr_factor(step):
        if step < warmup:
            return (step + 1) / warmup
        t = (step - warmup) / max(1, total_steps - warmup)
        return 0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * t))

    state = dict(epoch=0, step_in_epoch=0, global_step=0, best=-1.0, history=[])
    last = out_dir / "last.pt"
    if last.exists():
        ck = torch.load(last, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        scaler.load_state_dict(ck["scaler"])
        ema.shadow = {k: v.float().to(device) for k, v in ck["ema"].items()}
        state = ck["state"]
        print(f"▶ Продолжаю с эпохи {state['epoch']+1}, шаг {state['step_in_epoch']} (best={state['best']:.4f})")
    if state.get("done"):
        print("Модель уже обучена. Удалите last.pt чтобы обучить заново.")
        return out_dir

    weights = sample_weights(tr)
    ds = FundusDS(tr, data_dir, cfg["size"], train=True)
    t_start = time.time()
    last_ck = time.time()

    def save_last():
        atomic_save(dict(model=model.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(),
                         ema=ema.shadow, state=state, cfg=cfg), last)

    while state["epoch"] < cfg["epochs"]:
        ep = state["epoch"]
        sampler = EpochSampler(weights, steps_per_epoch * cfg["bs"] * cfg["accum"], ep,
                               start=state["step_in_epoch"] * cfg["bs"] * cfg["accum"])
        dl = DataLoader(ds, batch_size=cfg["bs"], sampler=sampler, num_workers=workers, pin_memory=True,
                        drop_last=True, persistent_workers=False, prefetch_factor=4 if workers else None)
        model.train()
        pbar = tqdm(dl, desc=f"[{key}] epoch {ep+1}/{cfg['epochs']}", initial=state["step_in_epoch"] * cfg["accum"],
                    total=steps_per_epoch * cfg["accum"])
        run = {}
        for it, (x, y) in enumerate(pbar):
            x = x.to(device, non_blocking=True)
            if cfg.get("channels_last") and device == "cuda":
                x = x.to(memory_format=torch.channels_last)
            y = y.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda" if device == "cuda" else "cpu", dtype=torch.float16,
                                enabled=device == "cuda"):
                out = model(x)
            loss, parts = multitask_loss(out, y, pos_weight)
            scaler.scale(loss / cfg["accum"]).backward()
            for k, v in parts.items():
                run[k] = 0.98 * run.get(k, v) + 0.02 * v
            if (it + 1) % cfg["accum"] == 0:
                f = lr_factor(state["global_step"])
                for g in opt.param_groups:
                    g["lr"] = g["base_lr"] * f
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                ema.update(model)
                state["global_step"] += 1
                state["step_in_epoch"] += 1
                pbar.set_postfix({k: f"{v:.3f}" for k, v in run.items()})
                if time.time() - last_ck > ckpt_minutes * 60:
                    save_last()
                    last_ck = time.time()
                if max_hours and time.time() - t_start > max_hours * 3600:
                    save_last()
                    print(f"⏸ Лимит времени {max_hours}ч — чекпоинт сохранён, перезапустите ячейку позже.")
                    return out_dir
        # ---- конец эпохи: валидация EMA-весов
        live = {k: v.detach().clone() for k, v in model.state_dict().items()}
        ema.copy_to(model)
        logits = predict(model, va, data_dir, cfg["size"], bs=cfg["bs"] * 2, tta=False, device=device, workers=workers)
        met = compute_metrics(logits, va[HEADS].values)
        met["epoch"] = ep + 1
        met["train_loss"] = {k: round(v, 4) for k, v in run.items()}
        state["history"].append(met)
        print(f"[{key}] epoch {ep+1}: " + ", ".join(f"{k}={v:.4f}" for k, v in met.items() if isinstance(v, float)))
        if met["score"] > state["best"]:
            state["best"] = met["score"]
            atomic_save(dict(model=model.state_dict(), cfg=cfg, metrics=met, key=key), out_dir / "best.pt")
            print(f"  ★ новый лучший score={met['score']:.4f} → best.pt")
        model.load_state_dict(live)
        state["epoch"] += 1
        state["step_in_epoch"] = 0
        save_last()
        last_ck = time.time()
        (out_dir / "history.json").write_text(json.dumps(state["history"], indent=1, default=float))
    state["done"] = True
    save_last()
    print(f"[{key}] обучение завершено. best score={state['best']:.4f}")
    return out_dir


def export_predictions(key: str, data_dir="/content/veya_data", drive_dir="/content/drive/MyDrive/veya_v2",
                       splits=("val", "test", "test_ext"), workers=2):
    """Предсказания лучшей модели (с TTA) + эмбеддинги → вход для MLP-«мозга»."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    out_dir = Path(drive_dir) / "models" / key
    ck = torch.load(out_dir / "best.pt", map_location="cpu", weights_only=False)
    model = build_model(key, pretrained=False)
    model.load_state_dict(ck["model"])
    model.to(device)
    df = pd.read_csv(Path(data_dir) / "labels.csv")
    pred_dir = Path(drive_dir) / "preds"
    pred_dir.mkdir(parents=True, exist_ok=True)
    for sp in splits:
        d = df[df.split == sp].reset_index(drop=True)
        if not len(d):
            continue
        r = predict(model, d, data_dir, ck["cfg"]["size"], bs=ck["cfg"]["bs"] * 2, tta=True,
                    device=device, return_emb=True, workers=workers)
        np.savez_compressed(pred_dir / f"{key}_{sp}.npz", image_id=d["image_id"].values, **r)
        met = compute_metrics(r, d[HEADS].values)
        print(f"[{key}] {sp}: " + ", ".join(f"{k}={v:.4f}" for k, v in met.items()))
    return pred_dir

## 1. Данные

In [ ]:
import hashlib, shutil, glob, time, pandas as pd
from veya_v2 import data_prep as D
COLAB_FP = 'cbbf7042b4391e204c38e14f2629368c'   # отпечаток сплита из Colab (125 760 снимков)
t0 = time.time()
L = D.run_all(work_dir=DATA, drive_dir=PARTS, delete_raw=True)
for t in glob.glob(f'{PARTS}/parts/*.tar'): os.remove(t)   # картинки уже лежат в DATA
shutil.copy(f'{PARTS}/labels.csv', f'{DATA}/labels.csv')
S = L.sort_values('image_id')
fp = hashlib.md5('\n'.join(S.image_id.astype(str) + ',' + S.split.astype(str)).encode()).hexdigest()
print(f'подготовка {(time.time()-t0)/60:.0f} мин;', len(L), L.split.value_counts().to_dict())
print('отпечаток сплита:', fp, '— СОВПАДАЕТ с Colab' if fp == COLAB_FP else '— ОТЛИЧАЕТСЯ от Colab')
shutil.copy(f'{DATA}/labels.csv', f'{OUT}/labels_kaggle.csv')

## 2. Обучение: ConvNeXt на GPU 0, ViT на GPU 1 (параллельно)

In [ ]:
import subprocess, textwrap
PLAN = {   # эпохи/сэмплы урезаны, чтобы обе модели + подготовка уложились в 12 ч Kaggle
    'convnext': dict(gpu=0, epochs=10, samples_per_epoch=24000),
    'vit':      dict(gpu=1, epochs=10, samples_per_epoch=24000),
}
procs = {}
for key, p in PLAN.items():
    script = f'''
import sys; sys.path.insert(0, "{CODE}")
from veya_v2 import train as T
T.MODELS["{key}"].update(epochs={p["epochs"]}, samples_per_epoch={p["samples_per_epoch"]})
T.train_model("{key}", data_dir="{DATA}", drive_dir="{OUT}", workers=2, ckpt_minutes=20)
T.export_predictions("{key}", data_dir="{DATA}", drive_dir="{OUT}", workers=2)
print("ГОТОВО {key}")
'''
    open(f'/tmp/train_{key}.py', 'w').write(script)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(p['gpu']), PYTHONUNBUFFERED='1')
    log = open(f'{OUT}/log_{key}.txt', 'w')
    procs[key] = subprocess.Popen([sys.executable, f'/tmp/train_{key}.py'], env=env, stdout=log, stderr=subprocess.STDOUT)
    print('запущено', key, 'на GPU', p['gpu'])

def tail(key, n=3):
    try:
        lines = [l for l in open(f'{OUT}/log_{key}.txt', errors='ignore').read().replace('\r', '\n').split('\n') if l.strip()]
        return ' | '.join(lines[-n:])[-400:]
    except Exception as e:
        return str(e)

t0 = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    print(f'--- {(time.time()-t0)/60:.0f} мин')
    for k in procs:
        print(f'[{k}]', 'работает' if procs[k].poll() is None else f'код {procs[k].returncode}', '::', tail(k))
for k, p in procs.items():
    print(k, 'код выхода', p.returncode)

## 3. Итог

In [ ]:
import json
for k in PLAN:
    h = f'{OUT}/models/{k}/history.json'
    if os.path.exists(h):
        hist = json.load(open(h)); best = max(hist, key=lambda m: m['score'])
        print(k, 'лучшая эпоха', best['epoch'], {m: round(v, 4) for m, v in best.items() if isinstance(v, float)})
    print(k, 'хвост лога:', tail(k, 8))
for root, _, files in os.walk(OUT):
    for f in files:
        p = os.path.join(root, f); print(round(os.path.getsize(p) / 1e6, 1), 'МБ', p)
for k in PLAN:   # last.pt не нужен для ансамбля — экономим место в Output
    lp = f'{OUT}/models/{k}/last.pt'
    if os.path.exists(lp) and os.path.exists(f'{OUT}/preds/{k}_val.npz'): os.remove(lp)